In [1]:
import numpy as np

np.set_printoptions(precision=3,suppress=True)

In [2]:
import sys
from os.path import dirname

sys.path.append(dirname("/home/azelenskiy/Documents/samosa/src/"))

In [3]:
from samosa.interactions import InteractionMap
from samosa.symmetry import point_group

## Introduction to `InteractionMap`

In [4]:
"""
InteractionMap class can be used to generate interactions matrices for simple
lattices ("chain", "square", "triangular", "cubic", "bcc", "fcc").

Currently, the method assumes that the patches are located at the centres of
particle faces.

Once generated, you can output the interaction map in face-pair or orientation
representation.
"""
# The general constructor uses the Schoenflies symbol for the site point group 
# (symmetry of the isotropic particle), as well as the location of a single 
# face. Below is the example for the triangular lattice
interaction_map = InteractionMap(point_group("C6"), [1,0,0])

# For simple lattices, use InteractionMap.from_lattice method
interaction_map = InteractionMap.from_lattice("triangular")

print(f"Interaction map in the bond-pair representation:\n{interaction_map.face_coupling_matrix}\n")

print(f"Interaction map in the orientation representation:\n{interaction_map.orientation_coupling_matrix}")


Interaction map in the bond-pair representation:
[[-0.664 -0.845 -0.191 -0.771 -0.463 -0.172]
 [-0.845  0.163  0.739  0.093 -0.732 -0.725]
 [-0.191  0.739 -0.444 -0.12   0.051 -0.953]
 [-0.771  0.093 -0.12   0.006  0.507 -0.432]
 [-0.463 -0.732  0.051  0.507  0.358 -0.361]
 [-0.172 -0.725 -0.953 -0.432 -0.361  0.359]]

Interaction map in the orientation representation:
[[-0.771 -0.463 -0.172 -0.664 -0.845 -0.191]
 [ 0.093 -0.732 -0.725 -0.845  0.163  0.739]
 [-0.12   0.051 -0.953 -0.191  0.739 -0.444]
 [ 0.006  0.507 -0.432 -0.771  0.093 -0.12 ]
 [ 0.507  0.358 -0.361 -0.463 -0.732  0.051]
 [-0.432 -0.361  0.359 -0.172 -0.725 -0.953]]


In [5]:
"""
When the InteractionMap object is initialized, the values are drawn from a 
Gaussian distribution (with mean 0.0 and std 1.0).

The matrix can be resampled or one can input a user-defined interaction map 
(make sure that the dimensions are correct and that the matrix is symmetric).
"""

# Input user-defined matrix 
n_faces = interaction_map.n_faces
matrix = 1.0 - 2.0 * np.eye(n_faces)

interaction_map.new_face_coupling_matrix(matrix)

print(f"Input interaction map:\n{interaction_map.face_coupling_matrix}\n")

# Resample interaction values
interaction_map.sample_face_coupling_matrix(j_mean=0.0, j_std=1.0)

print(f"Resampled interaction map:\n{interaction_map.face_coupling_matrix}\n")

Input interaction map:
[[-1.  1.  1.  1.  1.  1.]
 [ 1. -1.  1.  1.  1.  1.]
 [ 1.  1. -1.  1.  1.  1.]
 [ 1.  1.  1. -1.  1.  1.]
 [ 1.  1.  1.  1. -1.  1.]
 [ 1.  1.  1.  1.  1. -1.]]

Resampled interaction map:
[[ 0.858 -1.173  0.433  0.91  -0.569  0.197]
 [-1.173 -0.37  -0.379  0.08  -0.316 -0.783]
 [ 0.433 -0.379 -0.095 -0.25  -0.86   1.448]
 [ 0.91   0.08  -0.25   0.64   0.189  0.441]
 [-0.569 -0.316 -0.86   0.189 -0.524 -0.738]
 [ 0.197 -0.783  1.448  0.441 -0.738 -0.341]]



In [6]:
"""
InteractionMap object allows you to subtract the contribution that should be
directly related to the surface energy.
"""

print(f"Total interaction map:\n{interaction_map.face_coupling_matrix}\n")

print(f"Interaction map without surface contribution:\n{interaction_map.reduced_face_coupling_matrix}\n")

print(f"Surface contribution to the interaction map:\n{interaction_map.face_coupling_matrix - interaction_map.reduced_face_coupling_matrix}")


Total interaction map:
[[ 0.858 -1.173  0.433  0.91  -0.569  0.197]
 [-1.173 -0.37  -0.379  0.08  -0.316 -0.783]
 [ 0.433 -0.379 -0.095 -0.25  -0.86   1.448]
 [ 0.91   0.08  -0.25   0.64   0.189  0.441]
 [-0.569 -0.316 -0.86   0.189 -0.524 -0.738]
 [ 0.197 -0.783  1.448  0.441 -0.738 -0.341]]

Interaction map without surface contribution:
[[ 0.497 -0.935  0.131  0.323 -0.352 -0.092]
 [-0.935  0.467 -0.081  0.093  0.501 -0.473]
 [ 0.131 -0.081 -0.337 -0.777 -0.583  1.219]
 [ 0.323  0.093 -0.777 -0.173  0.181 -0.074]
 [-0.352  0.501 -0.583  0.181  0.272 -0.448]
 [-0.092 -0.473  1.219 -0.074 -0.448 -0.559]]

Surface contribution to the interaction map:
[[ 0.361 -0.238  0.302  0.587 -0.218  0.29 ]
 [-0.238 -0.837 -0.298 -0.012 -0.817 -0.31 ]
 [ 0.302 -0.298  0.242  0.527 -0.277  0.23 ]
 [ 0.587 -0.012  0.527  0.813  0.008  0.515]
 [-0.218 -0.817 -0.277  0.008 -0.796 -0.289]
 [ 0.29  -0.31   0.23   0.515 -0.289  0.218]]


## Suggested simulations

### Separate simulations for full and reduced interaction maps

To test if the implemented interaction map ensemble works, it would be useful to perform simulations using first the full interaction map $J_{ab}$, and then the reduced interaction map $J_{ab}^{(\text{red})}$.
The differences in the structures of the low-temperature aggregates should give us an indication of the usefulness of this ensemble.

### Linear interpolation between the reduced and full interaction maps

If the initial tests look promising, it may be useful to also look at interpolations of the interaction maps of the form

$$
J_{ab} - \alpha (J_{ab} - J_{ab}^{(\text{red})}),
$$
or 

$$
J_{ab} - \alpha J_{ab}^{(\text{red})},
$$
where $\alpha\in[0,1]$.
When $\alpha=1$, the first interpolation yields interactions without surface energy, whereas the second interpolation gives only the surface contributions.